# Brain Stroke CT — SIN393

Notebook inicial para executar no Google Colab e seguir o plano do projeto: auditoria do dataset, partição reproduzível e comparação de ResNet‑18/MobileNetV3‑Large com e sem augmentation.

**Uso acadêmico:** não é ferramenta diagnóstica. Não envie imagens clínicas privadas. Confira os termos/licença do Kaggle antes de usar os dados.


## 0. Runtime e acesso aos dados

Este notebook roda no runtime remoto do Colab. KaggleHub tenta baixar o dataset público sem credenciais; se o Kaggle exigir autenticação ou consentimento, salve `KAGGLE_API_TOKEN` em Secrets do Colab ou complete a etapa solicitada na conta Kaggle. Não cole tokens no notebook.


In [ ]:
%pip -q install kagglehub scikit-learn pandas pillow tqdm
import os, sys, platform, random, json, hashlib, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, torch, torchvision
SEED = 393
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'python':sys.version, 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'device':str(DEVICE), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


In [ ]:
# Baixa o dataset dentro do runtime Colab para não usar o hardware local.
import kagglehub
DATASET_SLUG = 'ozguraslank/brain-stroke-ct-dataset'
DATA_ROOT = Path('/content/brain-stroke-ct-dataset')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
download_path = kagglehub.dataset_download(DATASET_SLUG, output_dir=str(DATA_ROOT))
print('Dataset baixado em:', download_path)
print('Arquivos:', sum(p.is_file() for p in Path(download_path).rglob('*')))


## 1. Auditoria inicial

Primeiro confirme a estrutura e os rótulos retornados. Nenhum arquivo é descartado automaticamente. O mapa abaixo reconhece nomes comuns; revise-o com base na listagem real antes de prosseguir.


In [ ]:
from PIL import Image, UnidentifiedImageError
IMAGE_EXTS = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}
files = [p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
print('Total de imagens:',len(files))
print('Pastas de classe candidatas:', sorted({p.parent.name for p in files}))
print('Arquivos não imagem:',sum(p.is_file() and p.suffix.lower() not in IMAGE_EXTS for p in DATA_ROOT.rglob('*')))

def canonical_label(path):
    s = path.parent.name.lower().replace('_',' ').replace('-',' ')
    if any(x in s for x in ['hemorr','haemorr','bleed']): return 'hemorrhage'
    if any(x in s for x in ['ische','infarct','ischemic']): return 'ischemia'
    if any(x in s for x in ['normal','healthy','no stroke','non stroke','control']): return 'normal'
    return None

rows=[]; errors=[]; hashes={} 
for p in files:
    try:
        with Image.open(p) as im:
            w,h=im.size; mode=im.mode; im.verify()
        digest=hashlib.sha256(p.read_bytes()).hexdigest()
        rows.append({'path':str(p),'folder':p.parent.name,'label':canonical_label(p),'width':w,'height':h,'mode':mode,'sha256':digest})
        hashes.setdefault(digest,[]).append(str(p))
    except Exception as e: errors.append({'path':str(p),'error':str(e)})
audit=pd.DataFrame(rows)
print('Imagens legíveis:',len(audit),'| corrompidas/ilegíveis:',len(errors))
print('Dimensões mais comuns:\n',audit.groupby(['width','height']).size().sort_values(ascending=False).head(15))
print('Modos:',audit['mode'].value_counts().to_dict())
print('Contagens por pasta:\n',audit['folder'].value_counts())
print('Duplicatas exatas (grupos):',sum(len(v)>1 for v in hashes.values()))
display(audit[audit['label'].isna()][['path','folder']].head(30))
audit.to_csv('/content/auditoria_imagens.csv',index=False)
pd.DataFrame(errors).to_csv('/content/erros_leitura.csv',index=False)
assert len(audit)>0, 'Nenhuma imagem legível encontrada.'
assert audit['label'].notna().all(), 'Revise os rótulos/pastas desconhecidos e ajuste canonical_label antes de continuar.'
assert set(audit['label'].unique()) == {'normal','ischemia','hemorrhage'}, f'Classes observadas: {sorted(audit.label.unique())}; revise o mapeamento.'


## 2. Metadados e divisão

Inspecione nomes/arquivos CSV ou metadados do dataset para identificar paciente/estudo. Se houver identificadores, implemente GroupShuffleSplit/estratificação agrupada antes do treino. A divisão abaixo por imagem só é apropriada quando não há grupos identificáveis; a limitação deve ser reportada.


In [ ]:
metadata_files=[p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.tsv','.json','.xlsx'}]
print('Arquivos tabulares/metadados:',[str(p) for p in metadata_files])
for p in metadata_files:
    if p.suffix.lower() in {'.csv','.tsv'}:
        try: print('Amostra',p.name, pd.read_csv(p,sep='\t' if p.suffix.lower()=='.tsv' else ',').head(3).to_dict('records'))
        except Exception as e: print('Não foi possível ler',p.name,repr(e))
print('Exemplos de nomes de arquivo:',audit.path.head(20).map(lambda x:Path(x).name).tolist())
# Após revisar a estrutura, marque True somente se confirmou que não existem IDs
# de paciente/estudo. Se houver IDs, faça split agrupado antes de prosseguir.
CONFIRM_NO_PATIENT_OR_STUDY_IDS = False
assert CONFIRM_NO_PATIENT_OR_STUDY_IDS, 'Revise arquivos tabulares e nomes; implemente split agrupado se houver IDs. Depois confirme explicitamente ausência de grupos identificáveis.'
from sklearn.model_selection import train_test_split
valid = audit[audit.label.notna()].copy().reset_index(drop=True)
train_df, temp_df = train_test_split(valid, test_size=.30, random_state=SEED, stratify=valid.label)
val_df, test_df = train_test_split(temp_df, test_size=.50, random_state=SEED, stratify=temp_df.label)
train_df=train_df.copy(); train_df['split']='train'
val_df=val_df.copy(); val_df['split']='validation'
test_df=test_df.copy(); test_df['split']='test'
splits=pd.concat([train_df,val_df,test_df],ignore_index=True)
print(splits.groupby(['split','label']).size().unstack(fill_value=0))
splits[['path','label','split']].to_csv('/content/splits_seed393.csv',index=False)


## 3. Treino comparativo

Quatro cenários usam a mesma divisão. A seleção de checkpoint e early stopping usa macro‑F1 da validação; pesos de classe são calculados somente no treino. Augmentation ocorre apenas no treino, sem flips, com pequenas rotações/translações e brilho/contraste moderados. Revise resoluções, visualmente possíveis transformações e duração computacional antes de executar.


In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from collections import Counter
CLASSES=['normal','ischemia','hemorrhage']; LABEL_TO_ID={c:i for i,c in enumerate(CLASSES)}
SIZE=224; BATCH_SIZE=32 if torch.cuda.is_available() else 8; MAX_EPOCHS=12; PATIENCE=3
weights=models.ResNet18_Weights.DEFAULT
mean,std=weights.transforms().mean,weights.transforms().std
base=[transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)]
aug=[transforms.Resize((SIZE,SIZE)),transforms.RandomAffine(degrees=7,translate=(.03,.03)),transforms.ColorJitter(brightness=.12,contrast=.12),transforms.ToTensor(),transforms.Normalize(mean,std)]
class CTDataset(Dataset):
    def __init__(self, frame, augment=False): self.frame=frame.reset_index(drop=True); self.tf=transforms.Compose(aug if augment else base)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r.path) as im: x=self.tf(im.convert('RGB'))
        return x,LABEL_TO_ID[r.label]

def make_model(name):
    if name=='resnet18':
        m=models.resnet18(weights=models.ResNet18_Weights.DEFAULT); m.fc=nn.Linear(m.fc.in_features,3)
    elif name=='mobilenet_v3_large':
        m=models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT); m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,3)
    return m.to(DEVICE)

def evaluate(model, loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            p=model(x.to(DEVICE)).argmax(1).cpu().numpy(); ys.extend(y.numpy()); ps.extend(p)
    return f1_score(ys,ps,average='macro'), np.array(ys), np.array(ps)

def fit(name, augment):
    tr=DataLoader(CTDataset(train_df,augment),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
    va=DataLoader(CTDataset(val_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
    counts=Counter(train_df.label); class_w=torch.tensor([len(train_df)/(3*counts[c]) for c in CLASSES],dtype=torch.float32,device=DEVICE)
    model=make_model(name); criterion=nn.CrossEntropyLoss(weight=class_w); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
    best=-1; stale=0; history=[]; ckpt=f'/content/best_{name}_aug{int(augment)}.pt'
    for epoch in range(MAX_EPOCHS):
        start=time.time(); model.train(); total=0.
        for x,y in tr:
            x=x.to(DEVICE); y=y.to(DEVICE); optimizer.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); optimizer.step(); total+=loss.item()*len(y)
        vf1,_,_=evaluate(model,va); history.append({'epoch':epoch+1,'train_loss':total/len(train_df),'val_macro_f1':vf1,'seconds':time.time()-start})
        print(name,'augmentation=',augment,history[-1])
        if vf1>best: best=vf1; stale=0; torch.save(model.state_dict(),ckpt)
        else: stale+=1
        if stale>=PATIENCE: break
    return ckpt,pd.DataFrame(history)

# Execute all four planned conditions (uses only Colab runtime resources).
runs={}
for arch in ['resnet18','mobilenet_v3_large']:
    for aug_on in [False,True]: runs[(arch,aug_on)]=fit(arch,aug_on)


## 4. Avaliação final no teste

Execute apenas depois de congelar protocolo e hiperparâmetros. Esta célula avalia cada checkpoint selecionado pela validação no conjunto de teste e grava métricas, relatórios por classe e matrizes de confusão.


In [ ]:
test_loader=DataLoader(CTDataset(test_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
summary=[]; reports=[]
for (arch,aug_on),(ckpt,hist) in runs.items():
    model=make_model(arch); model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
    macro,ys,ps=evaluate(model,test_loader)
    report=classification_report(ys,ps,labels=list(range(3)),target_names=CLASSES,output_dict=True,zero_division=0)
    cm=confusion_matrix(ys,ps,labels=list(range(3)))
    summary.append({'architecture':arch,'augmentation':aug_on,'accuracy':accuracy_score(ys,ps),'macro_f1':macro,'weighted_f1':f1_score(ys,ps,average='weighted')})
    reports.append({'architecture':arch,'augmentation':aug_on,'report':report,'confusion_matrix':cm.tolist()})
    hist.to_csv(f'/content/history_{arch}_aug{int(aug_on)}.csv',index=False)
pd.DataFrame(summary).to_csv('/content/test_summary.csv',index=False)
Path('/content/test_details.json').write_text(json.dumps(reports,indent=2))
display(pd.DataFrame(summary)); print(json.dumps(reports,indent=2))


## Artefatos e reprodutibilidade

Salve em Drive apenas os relatórios necessários (split, auditoria, métricas e logs), sem imagens nem credenciais. Registre a versão exata do dataset, sua licença/termos, contagens observadas, hardware e eventuais limitações de divisão por paciente. Resultados não indicam desempenho clínico.
